# Laboratorio 5 — Control de Semáforos con Aproximación de Funciones

## Dominio

Una empresa de gestión de tráfico quiere usar RL con aproximación de función para controlar semáforos en una intersección con flujo variable. El estado es continuo: densidad de vehículos, velocidad promedio y tiempo de espera acumulado por carril.

Para tener números concretos, asumo una intersección de 4 accesos (N, S, E, O) con semáforo de dos fases (fase 1 = verde N-S, fase 0 = verde E-O). Por carril hay tres variables: $\rho_l$ (densidad), $v_l$ (velocidad promedio) y $w_l$ (espera acumulada), más $\text{fase}$ y $t_{fase}$ (tiempo desde el último cambio). Acciones: mantener o cambiar de fase. Recompensa: $-\sum_l w_l$ por paso. Es tarea continua (no episódica), así que se necesita $\gamma<1$.


## Pregunta 1 — Vector de características $x(s)$

Antes de armar $x(s)$ agrupo por eje, porque la fase le pega al mismo tiempo a N y S (o a E y O), así que no pierdo información relevante si trabajo con el promedio del eje en vez de carril por carril:

$$\rho_{NS}=\tfrac{\rho_N+\rho_S}{2},\ v_{NS}=\tfrac{v_N+v_S}{2},\ w_{NS}=\tfrac{w_N+w_S}{2}$$

y lo mismo para $\rho_{EO}, v_{EO}, w_{EO}$.

Con eso, propongo $x(s)\in\mathbb R^{12}$ y $\hat q(s,a;\mathbf w) = \mathbf w_a^\top x(s)$:

| $i$ | $x_i(s)$ | Rango | Por qué |
|---|---|---|---|
| 0 | $1$ (bias) | $\{1\}$ | intercepto |
| 1 | $\rho_{NS}/\rho_{max}$ | $[0,1]$ | congestión eje N-S |
| 2 | $\rho_{EO}/\rho_{max}$ | $[0,1]$ | congestión eje E-O |
| 3 | $1-v_{NS}/v_{max}$ | $[0,1]$ | qué tan lento va el eje N-S (densidad alta con velocidad normal no es lo mismo que densidad alta con velocidad casi cero) |
| 4 | $1-v_{EO}/v_{max}$ | $[0,1]$ | ídem eje E-O |
| 5 | $w_{NS}/w_{max}$ | $[0,1]$ | espera acumulada N-S, la variable más ligada a la recompensa |
| 6 | $w_{EO}/w_{max}$ | $[0,1]$ | espera acumulada E-O |
| 7 | $\text{fase}$ | $\{0,1\}$ | qué eje tiene verde ahora mismo |
| 8 | $t_{fase}/T_{max}$ | $[0,1]$ | qué tan avanzada va la fase actual |
| 9 | $\rho_{NS}\cdot\rho_{EO}/\rho_{max}^2$ | $[0,1]$ | interacción: ambos ejes saturados a la vez es peor que la suma de cada uno por separado |
| 10 | $\text{fase}\cdot w_{EO}/w_{max}$ | $[0,1]$ | espera que se está acumulando en el eje que está en rojo |
| 11 | $(1-\text{fase})\cdot w_{NS}/w_{max}$ | $[0,1]$ | lo mismo pero para N-S en rojo |

### ¿Alcanza?

$x_1$ a $x_8$ solas no bastan porque son puramente aditivas: no pueden expresar que el efecto de la espera de un eje depende de si ese eje está en rojo o verde en ese momento — eso es una interacción, no algo que una suma ponderada capture. Por eso agregué $x_9$–$x_{11}$ a mano como productos cruzados.

Aun así el vector se queda corto en un par de cosas: no captura la relación no lineal entre densidad y velocidad (en tráfico real la velocidad cae de golpe pasado cierto umbral de densidad, no de forma proporcional), ni efectos de saturación en $w$ (una espera muy larga probablemente pesa más que proporcionalmente en el costo real), ni interacciones de tres variables como fase×densidad×tiempo-en-fase. Para eso habría que meter términos cuadráticos o discretizar alguna variable (tile coding), no alcanza con seguir sumando cruces de a pares.


## Pregunta 2 — Parámetros: lineal vs. tabular

**Lineal:** con $d=12$ features y 2 acciones (un $\mathbf w_a$ por acción):

$$\#\text{parámetros} = 2\times 12 = 24$$

**Tabular:** el estado crudo tiene 12 variables continuas por carril ($\rho,v,w$ × 4 carriles) más $t_{fase}$ (continua) y fase (binaria) = 13 continuas + 1 binaria. Discretizando cada continua en 10 niveles:

$$\#\text{estados} = 10^{13}\times 2 = 2\times 10^{13}$$
$$\#\text{entradas } Q = 2\times 10^{13}\times 2\ (\text{acciones}) = 4\times 10^{13}$$

Eso es ~$10^{12}$ veces más parámetros que el modelo lineal. Incluso quedándome solo con las 2 densidades agregadas por eje (el caso más favorable para tabular) ya son $10^2\times 2 \times 2 = 400$ entradas.

**Consecuencia:** el modelo lineal generaliza — dos estados con densidades parecidas dan predicciones parecidas, así que una sola visita actualiza el valor de "vecinos" nunca vistos. La tabla no generaliza nada entre entradas: cada una de las $4\times10^{13}$ celdas es independiente y solo se actualiza si se visita directamente, lo cual es imposible con ese tamaño (curse of dimensionality) — en la práctica ni siquiera es entrenable. El modelo lineal paga eso con sesgo (solo puede representar lo que sus features permiten), pero con muchísima menos varianza y datos necesarios. Para este dominio, tabular no es "menos eficiente", es directamente inviable.


## Pregunta 3 — Por qué semi-gradiente TD no llega al mínimo de $J(\mathbf w)$

El objetivo que en teoría queremos minimizar es

$$J(\mathbf w) = \mathbb E_\pi\big[(V^\pi(S) - \hat V(S;\mathbf w))^2\big]$$

Un gradiente descendente de verdad necesitaría diferenciar completo respecto a $\mathbf w$. El problema es que no conocemos $V^\pi(S_t)$, así que TD lo reemplaza por el target bootstrapped $R_{t+1}+\gamma\hat V(S_{t+1};\mathbf w)$, que **también depende de $\mathbf w$**. La actualización de TD:

$$\mathbf w \leftarrow \mathbf w + \alpha\,\delta_t\,\nabla_{\mathbf w}\hat V(S_t;\mathbf w),\qquad \delta_t = R_{t+1}+\gamma\hat V(S_{t+1};\mathbf w)-\hat V(S_t;\mathbf w)$$

solo toma el gradiente de $\hat V(S_t;\mathbf w)$ y no del target — por eso es "semi"-gradiente y no gradiente completo. Al no derivar sobre el target, no está minimizando $J(\mathbf w)$ paso a paso: no hay garantía de que $\delta_t\nabla_{\mathbf w}\hat V(S_t;\mathbf w)$ apunte en la dirección que reduce $J$.

Lo que sí se puede mostrar es que, con aproximación lineal y entrenamiento on-policy, TD converge a un punto fijo $\mathbf w_{TD}$ distinto del que minimiza $J(\mathbf w)$: converge al punto donde el error TD proyectado sobre el espacio de features se hace cero, no al mínimo del error de valor real. Ese punto puede estar más lejos del $V^\pi$ real que la mejor aproximación lineal posible.

**Rol de $\gamma$:** la distancia entre el punto de TD y la mejor aproximación lineal posible crece con $\gamma$ — mientras más cerca de 1, más grave puede ser ese error (el factor de la cota es del orden de $1/(1-\gamma)$). Con $\gamma$ chico el punto de TD queda cerca del óptimo; con $\gamma\to1$ puede alejarse mucho.

Para semáforos esto importa porque uno quisiera $\gamma$ alto (una decisión de fase afecta la congestión varios ciclos después), pero justo ahí es donde el punto de TD se puede alejar más del valor real, sobre todo si las features no son perfectas (como discutimos en la P1, no lo son). Por eso conviene no irse a un $\gamma$ extremo tipo 0.99: algo como 0.9–0.95 balancea horizonte de planeación con la estabilidad de la aproximación.


## Pregunta 4 — Tríada mortal y ¿Deep RL o no?

La tríada mortal es: aproximación de función + bootstrapping + off-policy juntos, lo que puede causar divergencia.

En este dominio:
- **Aproximación de función**: obligatoria, ya vimos en la P2 que tabular ni siquiera es viable ($10^{13}$ estados).
- **Bootstrapping**: también casi obligatorio, porque es una tarea continua (no episódica) con recompensa densa — Monte Carlo puro no tiene mucho sentido sin episodios que terminen, y tendría varianza enorme por el horizonte largo.
- **Off-policy**: este es el único de los tres que es una decisión de diseño. Si se usa Q-learning se completa la tríada; con SARSA/Expected SARSA (on-policy) se rompe esa pata.

Entonces el componente más relevante para vigilar es el **off-policy**, porque es el único que se puede evitar — los otros dos vienen forzados por el dominio. La recomendación sería usar un algoritmo on-policy (SARSA semi-gradiente) para no completar la tríada, dado que no hay forma de quitar aproximación de función ni bootstrapping.

**¿Se justifica saltar a Deep RL?** Para una sola intersección, no lo veo justificado. La razón principal es que aunque el estado crudo es de alta dimensión (13 variables continuas), la estructura que realmente importa para decidir es baja: básicamente congestión por eje + fase + tiempo en fase, y eso se resume bien en las 12 features de la P1 sin necesitar que una red aprenda representaciones por sí sola. Además una red no elimina la tríada mortal (sigue usando aproximación de función + bootstrapping, y si es off-policy como DQN, sigue expuesta a inestabilidad) — solo la mitiga empíricamente con trucos como experience replay y target network, pero no hay garantía teórica de convergencia, así que tampoco es que resuelva el problema de fondo de la P3.

Donde sí tendría sentido Deep RL es si el alcance cambia: coordinar una red de varias intersecciones (ahí diseñar features e interacciones a mano ya no escala) o si el estado de entrada pasa a ser algo crudo tipo video de cámaras en vez de las variables ya procesadas (densidad/velocidad/espera). Para el caso de una intersección con estas variables ya medidas, aproximación lineal con buenas features + algoritmo on-policy debería bastar.
